# MiniCLIAgent 编码闭环 Demo

这份 notebook 用来演示一个最小的编码闭环：

1. 先准备一个小 bug
2. 让 agent 读取文件并修改
3. 再验证修改结果

它的目标不是做复杂开发，而是让学习者看到一个真实的 `read -> edit -> verify` 流程。

## 0. 开始前

请确认：

- 当前目录是仓库根目录
- `.env` 已配置
- `MINICLIAGENT_WORKSPACE` 已指向一个可写工作区
- 你愿意让 agent 修改这个工作区中的测试文件

In [1]:
from pathlib import Path

workspace = Path.cwd() / ".demo-workspace"
workspace.mkdir(exist_ok=True)
(workspace / "sample.py").write_text(
    "def normalize_name(text: str) -> str:\n"
    "    return text\n"
)
print(workspace)
print((workspace / "sample.py").read_text())

/Users/yuanzilin/Minicliagent/docs/getting-started/.demo-workspace
def normalize_name(text: str) -> str:
    return text



上面这一步创建了一个非常小的示例文件。当前实现有一个明显问题：它没有去掉空白，也没有转成小写。

In [ ]:
import os
os.environ["MINICLIAGENT_WORKSPACE"] = str(workspace)

In [ ]:
!python -m minicliagent.cli.main run --session coding-demo --prompt "Use tools if needed. Read sample.py, change normalize_name so it returns text.strip().lower(), and reply with exactly DONE when finished."

如果这一步成功，agent 应该已经完成了一次最小文件修改。

In [ ]:
print((workspace / "sample.py").read_text())

In [ ]:
namespace = {}
exec((workspace / "sample.py").read_text(), namespace)
result = namespace["normalize_name"]("  Alice  ")
print(result)
assert result == "alice"

现在你已经走完了一次最小编码闭环：

- 准备一个问题
- 让 agent 读文件
- 让 agent 改文件
- 再用代码验证结果

这个流程比“只看 agent 回复”更重要，因为它体现的是工程验证，而不是表面成功。